## Strategies de chunking avancées

1. **CharacterTextSplitter** : Découpage basique par caractères
2. **RecursiveCharacterTextSplitter** : Découpage hierarchique
3. **MarkdownTextSplitter** : Découpage en respectant la structure Markdown

In [1]:
# Installation
# !uv add langchain-text-splitters sentence-transformers chromadb

In [2]:
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    MarkdownTextSplitter,
)
import numpy as np
import pandas as pd

c:\Users\Administrateur\Documents\M2i_CDSD_TDTP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# préparation des données
rag_article = """
# Retrieval-Augmented Generation (RAG) : Guide Complet

## Introduction au RAG

Le Retrieval-Augmented Generation (RAG) est une technique qui combine la recuperation d'informations avec la generation de texte par des modeles de langage. Cette approche permet de surmonter les limitations des LLMs traditionnels en leur donnant acces a des connaissances externes et a jour.

Les LLMs classiques comme GPT-4 ou Claude sont entraines sur des donnees statiques. Ils ne peuvent pas acceder a des informations posterieures a leur date d'entrainement, ni a des donnees privees ou specifiques a une entreprise. Le RAG resout ce probleme en recuperant dynamiquement les informations pertinentes avant la generation.

## Architecture du RAG

L'architecture RAG se compose de trois composants principaux :

### 1. Le Systeme de Recuperation (Retrieval)

Le systeme de recuperation est responsable de trouver les documents pertinents dans une base de connaissances. Il utilise generalement des embeddings vectoriels pour representer les documents et les requetes dans un espace semantique. Les techniques courantes incluent :

- **Recherche par similarite cosinus** : Compare les vecteurs de la requete et des documents
- **Maximum Marginal Relevance (MMR)** : Equilibre pertinence et diversite des resultats
- **Hybrid Search** : Combine recherche semantique et recherche par mots-cles (BM25)

### 2. La Base Vectorielle (Vector Store)

La base vectorielle stocke les documents sous forme d'embeddings. Les options populaires incluent :

- **ChromaDB** : Base vectorielle open-source, facile a utiliser
- **Pinecone** : Solution cloud scalable
- **Weaviate** : Base vectorielle avec capacites de recherche hybride
- **FAISS** : Bibliotheque Facebook pour recherche de similarite rapide

### 3. Le Modele de Generation (LLM)

Le LLM genere la reponse finale en s'appuyant sur les documents recuperes. Les modeles couramment utilises sont :

- **GPT-4** : Excellent pour des reponses nuancees
- **Claude** : Bon pour des textes longs
- **Llama 2** : Option open-source performante
- **Mistral** : Modele francais avec bon rapport performance/taille

## Le Chunking : Etape Cruciale

Le chunking est le processus de decoupage des documents en morceaux plus petits. C'est une etape critique qui impacte directement la qualite du RAG.

### Pourquoi le Chunking est Important

Un bon chunking permet de :
1. Respecter les limites de contexte des LLMs
2. Ameliorer la precision de la recherche
3. Reduire le bruit dans les resultats
4. Optimiser les couts (moins de tokens envoyes au LLM)

### Le Dilemme Taille des Chunks

Il faut trouver le bon equilibre :

**Chunks trop petits** :
- Perte de contexte
- Fragments incomplets
- Moins de sens semantique

**Chunks trop grands** :
- Bruit dans les resultats
- Risque de depasser le contexte du LLM
- Recherche moins precise

### Strategies de Chunking

#### 1. Fixed-Size Chunking

Decoupe tous les X caracteres, sans tenir compte de la structure. Simple mais brutal.

#### 2. Recursive Chunking

Essaie de decouper selon une hierarchie de separateurs :
1. Double saut de ligne (paragraphes)
2. Simple saut de ligne
3. Fin de phrase
4. Virgule
5. Espace

C'est la methode recommandee pour la plupart des cas.

#### 3. Semantic Chunking

Utilise des embeddings pour detecter les ruptures semantiques. Coupe quand la similarite entre phrases consecutives chute brutalement. Plus intelligent mais plus lent.

#### 4. Document-Aware Chunking

Respecte la structure du document (headers Markdown, fonctions Python, etc.). Ideal pour du code ou de la documentation structuree.

## Techniques Avancees de Retrieval

### Maximum Marginal Relevance (MMR)

MMR equilibre pertinence et diversite avec la formule :

```
MMR = λ * Sim(q, d) - (1-λ) * max Sim(d, d_i)
```

Ou :
- λ controle l'equilibre (0 = diversite, 1 = pertinence)
- Sim(q, d) est la similarite entre query et document
- max Sim(d, d_i) mesure la similarite avec les docs deja selectionnes

### Hybrid Search

Combine deux approches complementaires :

1. **BM25** (keyword-based) : Excellent pour les termes techniques, noms propres, acronymes
2. **Vector Search** (semantic) : Capture le sens, gere les synonymes et reformulations

La fusion des scores se fait generalement avec Reciprocal Rank Fusion (RRF).

### Self-Query Retriever

Utilise un LLM pour analyser la requete et extraire :
- La partie semantique (pour la recherche vectorielle)
- Les filtres de metadonnees (date, categorie, auteur, etc.)

Exemple : "Articles sur le ML publies en 2024" devient :
- Recherche : "machine learning"
- Filtre : year = 2024

## Reranking : Ameliorer la Precision

Le reranking est une etape de post-traitement qui reordonne les documents recuperes pour maximiser la pertinence.

### Bi-encoder vs Cross-encoder

**Bi-encoder** (utilisé pour le retrieval initial) :
- Encode query et documents separement
- Compare les embeddings (cosinus)
- Rapide (embeddings pre-calcules)
- Moins precis

**Cross-encoder** (utilisé pour le reranking) :
- Encode [query + document] ensemble
- Attention croisee complete
- Plus lent (pas de pre-calcul possible)
- Beaucoup plus precis

### Pipeline Optimal

1. **Retrieval** avec bi-encoder : Recuperer top-20 candidats (rapide)
2. **Reranking** avec cross-encoder : Reordonner et garder top-4 (precis)
3. **Generation** avec LLM : Generer la reponse avec les 4 meilleurs docs

## Optimisations et Best Practices

### Chunk Overlap

Toujours utiliser un overlap (10-20% du chunk_size) pour preserver le contexte aux frontieres :

```python
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200  # 20% overlap
)
```

### Enrichissement des Metadonnees

Ajouter des metadonnees riches pour faciliter le filtrage :
- Source du document
- Date de creation/modification
- Auteur
- Categorie/tags
- Position du chunk dans le document
- Taille du chunk

### Parent-Child Chunking

Strategie avancee :
1. Creer des petits chunks pour la recherche (precision)
2. Stocker des grands chunks parents (contexte)
3. Rechercher via les petits chunks
4. Retourner les parents au LLM

Meilleur des deux mondes : precision de recherche + contexte riche.

## Conclusion

Le RAG est une technique puissante mais complexe. Les facteurs cles de succes sont :

1. **Chunking adapte** au type de documents
2. **Retrieval robuste** avec techniques avancees (MMR, hybrid)
3. **Reranking** pour maximiser la pertinence
4. **Evaluation continue** de la qualite des reponses

Les parametres optimaux dependent fortement du use case. Il est essentiel de tester et mesurer sur des donnees representatives.
"""

print(f"Longueur du document : {len(rag_article)} caracteres")
print(f"Nombre de mots : {len(rag_article.split())} mots")

Longueur du document : 6626 caracteres
Nombre de mots : 1006 mots


### Strategie 1 : CharacterTextSplitter

In [4]:
character_splitter = CharacterTextSplitter(
    separator="\n", chunk_size=500, chunk_overlap=50, length_function=len
)

character_chunks = character_splitter.split_text(rag_article)

print(f"Nombre de chunks : {len(character_chunks)}")
print(f"Exemple de chunk :")
print(character_chunks[0])

Nombre de chunks : 15
Exemple de chunk :
# Retrieval-Augmented Generation (RAG) : Guide Complet
## Introduction au RAG
Le Retrieval-Augmented Generation (RAG) est une technique qui combine la recuperation d'informations avec la generation de texte par des modeles de langage. Cette approche permet de surmonter les limitations des LLMs traditionnels en leur donnant acces a des connaissances externes et a jour.


## Strategie 2 : RecursiveCharacterTextSplitter

In [5]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=50, separators=["\n\n", "\n", ". ", ", ", " ", ""]
)

recursive_chunks = recursive_splitter.split_text(rag_article)

print(f"Nombre de chunks : {len(recursive_chunks)}")
print(f"Exemple de chunk :")
print(len(recursive_chunks[0]))


Nombre de chunks : 16
Exemple de chunk :
372


## Strategie 3 : MarkdownTextSplitter

In [6]:
markdow_splitter = MarkdownTextSplitter(chunk_size=500, chunk_overlap=50)

markdown_chunks = markdow_splitter.split_text(rag_article)

print(f"Nombre de chunks : {len(markdown_chunks)}")
print(f"Exemple de chunk :")
print(len(markdown_chunks[0]))

Nombre de chunks : 19
Exemple de chunk :
372
